# School location: which zone a high school student attends

## What this model answers

Which zone does a high school student (grades 9-12) go to school in? The model compares all zones with high enrolment by size, car time from home, boundaries and school attributes. Home and school are known only as survey zones or tracts, so both are spread over several model zones.

| | |
|---|---|
| Choice | The school zone |
| Alternatives | All zones with high enrolment (full choice set) |
| Decision makers | High school students |
| Data | Regional Travel Survey 2017/18 (Baltimore and Washington regions) |
| Estimation sample | 668 students |
| Hold-out sample | 667 students |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3); the data rebuild also needs `python3` with geopandas. Then *Run All*.
- `RUN_DATA_PREP`: rebuild the estimation tables in a scratch folder and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the article coefficients, no estimation (about 3.2 min).
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default; the results in `output/` are read instead) — several hours for this model.
- `RUN_HOLDOUT`: log-likelihood of the hold-out records.
- Figures of the hold-out check are read from `output/holdout_*` (copied there by the notebook builder).

## Contents

Article table: `est_school (high school panel)`. This notebook goes from the raw records to the estimates of this model:

1. Raw data
2. Data preparation: filters from the raw records to the estimation sample (records kept and dropped), rebuild check
3. Estimation data and choice set
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_school_location_high_school.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

11. Glossary

## Model at a glance

**1. Who decides.** High school students. Data: Regional Travel Survey 2017/18 (Baltimore and Washington regions). Estimation sample: 668 students; hold-out: 667.

**2. What is chosen (dependent variable).** The school zone. Alternatives: All zones with high enrolment (full choice set).

**3. Explanatory variables** (signs of the article estimates, `model/reference_estimates.csv`):

| Group | Variable | What it measures | Sign |
|---|---|---|---|
| Size | Size coefficient mu | ln of the zone's weighted enrolment | + |
| Size | In-boundary public enrolment (log weight) | against out-of-boundary public | + |
| Car time | Car time (min) | AM-peak car minutes home to school zone | − |
| Boundaries | Other county x public share | public share of the zone's enrolment, other county | − |
| School zone | Charter-or-magnet share | share of the zone's enrolment | + |
| School zone | Own-race share | share of the zone's enrolment of the student's race category | + |

**4. How household traits enter.** Household traits (income, size, age, cars, children) are the same for every alternative, so on their own they cannot change which alternative is chosen. They enter only as interactions with attributes of the alternatives (for example density x single-person household, car time x income).

**5. How to read a coefficient.** A positive coefficient makes the alternative more likely, a negative one less likely. exp(coefficient) is the factor by which the odds change for one unit more of the variable.
 Example: Car time (min) = −0.103, odds factor exp(-0.103) = 0.902 per unit.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the estimation sample and tables into output/rebuild_check and compare with the shipped tables (section 2)
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)  (this model: several hours and more than 8 GB)
RUN_HOLDOUT    = True    # log-likelihood of the hold-out sample (at your estimates if there are any, else at the article coefficients)

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import biogeme.biogeme as bio
from biogeme.parameters import Parameters
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg
fg = importlib.reload(fg)
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "school_location_high_school"; MODEL_FOLDER = "07_school_location_high_school"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

In [ ]:
# Raw data
SH = os.path.abspath(os.path.join(FOLDER, "..", "00_shared_data")); CODE = os.path.join(FOLDER, "input", "code")
RAW = os.path.join(FOLDER, "input", "data", "raw"); REBUILD = os.path.join(OUT, "rebuild_check")
ZS = pd.read_csv(os.path.join(SH, "zone_system", "zoneSystem.csv"))[["ZoneId", "COUNTYFIPS", "JUR_NAME", "PUMA_10"]].rename(columns={"ZoneId": "zone"})
ZS["area_group"] = nt.area_group_of_county(ZS.COUNTYFIPS)

def rebuild_tables(python=sys.executable, pre=""):
    """build_estimation_data.main() of input/code with its output folder set to a scratch folder; returns the comparison with the
    shipped estimation and hold-out tables."""
    out = os.path.join(REBUILD, "data"); [os.makedirs(os.path.join(out, h), exist_ok=True) for h in ("estimation", "holdout")]
    nt.run(f'{python} -c "import build_estimation_data as B; {pre}B.OUT = {out!r}; B.main()"', cwd=CODE)
    return pd.concat([nt.compare_trees(os.path.join(out, h), os.path.join(FOLDER, "input", "data", h)).assign(half=h) for h in ("estimation", "holdout")],
                     ignore_index=True)

### 1.1 The 2017/18 Regional Travel Survey (students)

The school location models use the students of the 2017/18 RTS with their home and school locations (survey traffic zone or
tract). `../00_shared_data/school_choice_work/students.parquet` holds one row per student of the three segments (K-8,
high school, college) with the modal home zone `hz`, school zone `sz` and the half of the household split.

In [ ]:
# 1.1 The 2017/18 Regional Travel Survey (students)
flat = pd.read_csv(os.path.join(SH, "rts_2017_18", "RTS_raw_flat_data.csv"), low_memory=False, usecols=lambda c: not nt.RACE.search(c))
per = flat.drop_duplicates("person_id")
STU = pd.read_parquet(os.path.join(SH, "school_choice_work", "students.parquet"))
print(f"persons of the survey: {len(per):,}; students in students.parquet: {len(STU):,}")
display(STU.groupby(["seg", "half"]).size().unstack())
STU[STU.seg == 'hs'].head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From survey persons to students

`sc_01_data.py` keeps students (`student_status` 1) of a school type with an inventory of schools: K-8 (type 2), 9th-12th
(type 3), college (2-year, 4-year, graduate: types 6-8); daycare, home school, vocational and other schools are left out.
It drops online-only students, ages outside the segment range (hs: 13-19), students whose home or school is not located
in a zone with enrolment of their band, and households without income. The first filters are mirrored below; the last
ones need the location step of `sc_01_data.py`, which is not run here (it reads the survey through the original project
folders and writes there), so the result is taken from the shipped `students.parquet`.

In [ ]:
# Data preparation
SEGMAP = {2: "k8", 3: "hs", 6: "col", 7: "col", 8: "col"}
f = nt.Funnel("persons of the survey", len(per))
x = per[per.student_status == 1]; f.add("students", len(x), "not a student")
x = x[x.school_type.isin(SEGMAP)]; f.add("school type with a school inventory", len(x), "daycare, home school, vocational, other")
x = x[x.school_type.map(SEGMAP) == 'hs']; f.add("segment hs", len(x), "other segments")
x = x[x.school_freq != 8]; f.add("not online only", len(x), "online classes only")
lo, hi = {"k8": (5, 14), "hs": (13, 19), "col": (16, 200)}['hs']
x = x[x.age.between(lo, hi)]; f.add("age in the segment range", len(x), "age missing or outside the range")
s_ = STU[STU.seg == 'hs']
f.add("home and school located, income known (students.parquet)", len(s_), "home or school not in a zone with enrolment of the band; income missing")
f.add("estimation half", int((s_.half == "estimation").sum()), "hold-out half")
display(f.table()); nt.funnel_plot(f, "persons")
print("students of students.parquet among the mirrored records:", bool(set(s_.person_id) <= set(x.person_id)))

### 2.2 Rebuild check

`build_estimation_data.main()` (full choice set of the zones with enrolment of the band, location weights of home and school) runs with a scratch output folder; the student, alternative and candidate tables of both halves are compared with the shipped ones.

In [ ]:
# 2.2 Rebuild check
if RUN_DATA_PREP:
    t0 = time.time(); res = rebuild_tables(); print(f"build: {(time.time() - t0) / 60:.1f} min"); display(res)
    nt.rebuild_verdict(res, glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False)")

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**Regional Travel Survey 2017/18 (household, person, trip records)** — Baltimore Metropolitan Council (BMC) and National Capital Region Transportation Planning Board (TPB/MWCOG)
- Where: https://www.baltometro.org
- How: Restricted: request the 2017/2018 Regional Travel Survey data from BMC / TPB. File used: RTS_raw_flat_data.csv (flat person-level file with household fields: household_id, person_id, home_state_county_fips, home_tract_fips, home_type, hhsize, numvehicle, numdrivers, hh_income_detailed, wthhfin, wtperfin, age, age_group, employment_status, j1_workplace_loc, work_state_county_fips, work_tract_fips, license, school locations and BMC/TPB TAZ codes). Also rts_trips_clean.csv (cleaned trips) for the synthetic-population workplace step.
- Put in: 00_shared_data/raw/rts_2017_18/
- Files: `RTS_raw_flat_data.csv`, `rts_trips_clean.csv`

**MSTM zone system, zone polygons and peak car skim (Maryland Statewide Transportation Model)** — Maryland Department of Transportation State Highway Administration (MDOT SHA), provided to the project
- Where: https://roads.maryland.gov
- How: Not a public download: the 1,588-zone MSTM system (zoneSystem.csv), zone polygons (SMZ_RMZ_02152011inMSTM_EPSG26918.shp) and the AM-peak auto travel-time skim were provided by MDOT. Request them from MDOT SHA (MSTM / travel forecasting).
- Put in: 00_shared_data/raw/mstm/
- Files: `zoneSystem.csv`, `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.shx/.dbf/.prj`, `auto_peak_time.csv.gz`

**NCES school directories: CCD 2024-25, EDGE geocodes, Private School Survey 2023-24** — National Center for Education Statistics
- Where: https://nces.ed.gov/ccd/Data/zip/ccd_sch_029_2425_w_1a_073025.zip
- How: Download ccd_sch_029_2425_w_1a_073025.zip, ccd_sch_052_2425_l_1a_073025.zip, ccd_sea_052_2425_l_1a_073025.zip, ccd_lea_029_2425_w_1a_073025.zip (https://nces.ed.gov/ccd/Data/zip/), EDGE_GEOCODE_PUBLICSCH_2425.zip and EDGE_GEOCODE_PRIVATESCH_2324.zip (https://nces.ed.gov/programs/edge/data/), pss2324_pu_csv.zip (https://nces.ed.gov/surveys/pss/zip/pss2324_pu_csv.zip).
- Put in: 00_shared_data/raw/nces/
- Files: `ccd_sch_029_2425_w_1a_073025.zip`, `ccd_sch_052_2425_l_1a_073025.zip`, `ccd_sea_052_2425_l_1a_073025.zip`, `ccd_lea_029_2425_w_1a_073025.zip`, `EDGE_GEOCODE_PUBLICSCH_2425.zip`, `EDGE_GEOCODE_PRIVATESCH_2324.zip`, `pss2324_pu_csv.zip`

**IPEDS HD 2024, fall enrolment EF2023A, 12-month enrolment EFFY2024** — National Center for Education Statistics (IPEDS)
- Where: https://nces.ed.gov/ipeds/datacenter/data/HD2024.zip
- How: Download HD2024.zip, EF2023A.zip, EFFY2024.zip and their _Dict.zip from https://nces.ed.gov/ipeds/datacenter/data/.
- Put in: 00_shared_data/raw/ipeds/
- Files: `HD2024.zip`, `EF2023A.zip`, `EFFY2024.zip`

**School Attendance Boundary Survey 2015-16** — NCES EDGE
- Where: https://nces.ed.gov/programs/edge/data/SABS_1516_SchoolLevels.zip
- How: Download and unzip into ex/ (layers SABS_1516_Primary, _Middle, _High, _Other). Page https://nces.ed.gov/programs/edge/sabs.
- Put in: 00_shared_data/raw/sabs/
- Files: `SABS_1516_SchoolLevels.zip`

**BMC TAZ 2010 and TPB TAZ 3722 boundaries (the TAZ codes of the RTS)** — Baltimore Metropolitan Council; MWCOG / TPB
- Where: https://gis.baltometro.org/arcgis/rest/services/Regional_Boundaries/MapServer/14
- How: ArcGIS REST query (where=1=1, outFields=*, outSR=26918, f=geojson) of BMC layer 14 -> layer14.geojson; TPB layer https://gis.mwcog.org/wa/rest/services/RTDC/TAZ/MapServer/0/query in pages of 1,000 -> rest_layer0_off{0,1000,2000,3000}.geojson.
- Put in: 00_shared_data/raw/taz/
- Files: `layer14.geojson`, `rest_layer0_off0.geojson ... off3000.geojson`

**Census 2010 Redistricting (PL 94-171) Maryland file and TIGER 2010 blocks and tracts** — U.S. Census Bureau
- Where: https://www2.census.gov/census_2010/01-Redistricting_File--PL_94-171/Maryland/
- How: md2010.pl.zip (geo file and segment 2, housing units H1) and TIGER/Line 2010 tl_2010_24_tabblock10.zip; tracts tl_2010_<st>_tract10.zip for st = 10, 11, 24, 42, 51, 54 (https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/).
- Put in: 00_shared_data/raw/census2010/
- Files: `md2010.pl.zip`, `tl_2010_24_tabblock10.zip`, `tl_2010_<st>_tract10.zip`

**Census 2020 Redistricting (PL 94-171) files, DC, VA, WV** — U.S. Census Bureau
- Where: https://www2.census.gov/programs-surveys/decennial/2020/data/01-Redistricting_File--PL_94-171/<State>/<st>2020.pl.zip
- How: Download dc2020.pl.zip, va2020.pl.zip, wv2020.pl.zip (geo file summary level 750, table H1).
- Put in: 00_shared_data/raw/census2020/
- Files: `dc2020.pl.zip`, `va2020.pl.zip`, `wv2020.pl.zip`

**LEHD LODES 8, Workplace Area Characteristics 2016 and geography crosswalk, Maryland** — U.S. Census Bureau, LEHD
- Where: https://lehd.ces.census.gov/data/lodes/LODES8/md/wac/md_wac_S000_JT00_2016.csv.gz
- How: Download md_wac_S000_JT00_2016.csv.gz and https://lehd.ces.census.gov/data/lodes/LODES8/md/md_xwalk.csv.gz (other states: .../LODES8/<st>/wac/<st>_wac_S000_JT00_<year>.csv.gz). Workplace zone attributes (high-earning share CE03/C000) use the 2017 WAC files of md, dc, va, pa, de, wv: https://lehd.ces.census.gov/data/lodes/LODES8/<st>/wac/<st>_wac_S000_JT00_2017.csv.gz (project: data/raw/lodes_wac/ and data/raw/lodes_wac_2017_extra/).
- Put in: 00_shared_data/raw/lodes_wac/
- Files: `md_wac_S000_JT00_2016.csv.gz`, `md_xwalk.csv.gz`, `<st>_wac_S000_JT00_2017.csv.gz (st = md, dc, va, pa, de, wv)`

**BLS JOLTS job-openings rate, Maryland** — U.S. Bureau of Labor Statistics
- Where: https://api.bls.gov/publicAPI/v1/timeseries/data/JTS000000240000000JOR
- How: BLS public API v1, series JTS000000240000000JOR (Maryland) and JTS000000000000000JOR (U.S.), annual mean of monthly SA values.
- Put in: 00_shared_data/raw/jolts/
- Files: `jolts_bls_api.json`

**ACS PUMS 2014-2018 5-year, household and person files** — U.S. Census Bureau
- Where: https://www2.census.gov/programs-surveys/acs/data/pums/2018/5-Year/
- How: Download csv_h<st>.zip and csv_p<st>.zip for st = dc, de, md, pa, va, wv from the 2018 5-Year folder (CSV, not SAS). Variables used: SERIALNO, ST, PUMA, MIGPUMA, MIGSP, POWPUMA, POWSP, MIG, MV, RELP, AGEP, ESR, JWMNP, SCHG, NP, TYPE, BDSP, BLD, TEN, GRNTP, SMOCP, VALP, HINCP, VEH, YBL, WGTP; ADJINC (income adjustment) and ADJHSG (housing-cost adjustment) are columns of the same files.
- Put in: 00_shared_data/raw/acs_pums_2014_2018_5yr/
- Files: `csv_hdc.zip`, `csv_hde.zip`, `csv_hmd.zip`, `csv_hpa.zip`, `csv_hva.zip`, `csv_hwv.zip`, `csv_pdc.zip`, `csv_pde.zip`, `csv_pmd.zip`, `csv_ppa.zip`, `csv_pva.zip`, `csv_pwv.zip`

**School test results of Maryland public schools (state assessment, % proficient), 2016-17 and 2018-19** — U.S. Department of Education EDFacts, through the Urban Institute Education Data Portal
- Where: https://educationdata.urban.org/api/v1/schools/edfacts/assessments/2016/grade-99/?fips=24
- How: Download the JSON of https://educationdata.urban.org/api/v1/schools/edfacts/assessments/2016/grade-99/?fips=24 and of the same address with 2018 (school-level results, all grades, all students); save them as edfacts_assessments_md_2016_grade99.json and edfacts_assessments_md_2018_grade99.json. The Maryland Report Card download page returned server errors when the data were collected.
- Put in: 00_shared_data/school_attributes_raw/msde_reportcard/
- Files: `edfacts_assessments_md_2016_grade99.json`, `edfacts_assessments_md_2018_grade99.json`

**NCES CCD 2024-25 membership by race and grade (052) and directory (029, charter flag); CCD 2017-18 school characteristics (129, magnet flag); Private School Survey 2023-24** — National Center for Education Statistics
- Where: https://nces.ed.gov/ccd/Data/zip/ccd_sch_129_1718_w_1a_083118.zip
- How: Download ccd_sch_052_2425_l_1a_073025.zip and ccd_sch_029_2425_w_1a_073025.zip (https://nces.ed.gov/ccd/Data/zip/), ccd_sch_129_1718_w_1a_083118.zip (same folder) and pss2324_pu_csv.zip (https://nces.ed.gov/surveys/pss/zip/pss2324_pu_csv.zip).
- Put in: 00_shared_data/school_attributes_raw/nces/
- Files: `ccd_sch_052_2425_l_1a_073025.zip`, `ccd_sch_029_2425_w_1a_073025.zip`, `ccd_sch_129_1718_w_1a_083118.zip`, `pss2324_pu_csv.zip`

**IPEDS fall enrolment by race 2023 (EF2023A) and institutional characteristics 2023-24 (IC2023: dormitory capacity)** — National Center for Education Statistics (IPEDS)
- Where: https://nces.ed.gov/ipeds/datacenter/data/IC2023.zip
- How: Download EF2023A.zip, IC2023.zip and their _Dict.zip from https://nces.ed.gov/ipeds/datacenter/data/.
- Put in: 00_shared_data/school_attributes_raw/ipeds/
- Files: `EF2023A.zip`, `IC2023.zip`

## 3. Estimation data and choice set

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Alternatives: all zones with high-school enrolment; availability = enrolment. Home and school locations are mixtures over up to 3 home and 2 school zones (`w_home_<h>`, `w_school_<c>`).

The table exactly as used in the estimation, and its data dictionary.

In [ ]:
# Estimation data and choice set
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv"))
dd

In [ ]:
# Estimation data and choice set
import estimate_school_location_high_school as M
st = pd.read_parquet(f"{M.DATA}_students.parquet")          # one row per student: person attributes, possible home and school zones
alt = pd.read_parquet(f"{M.DATA}_alternatives.parquet")     # sampled school zones of each possible home zone
cand = pd.read_parquet(f"{M.DATA}_candidates.parquet")      # the possible chosen school zones
for k, t in [("students", st), ("alternatives", alt), ("candidates", cand)]:
    print(f"{k:13s} {t.shape[0]:>10,} rows  {t.shape[1]:>3} columns")
display(st.head()); display(alt.head())

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

In [ ]:
# Car time from home to the chosen school (weighted over the possible home and school zones), area of the school
CAND = cand.merge(st[["student"]], on="student")
w = st.set_index("student")
c = CAND.copy()
c["wh"] = [w.at[s_, f"w_home_{h}"] for s_, h in zip(c.student, c.home_slot)]
c["ws"] = [w.at[s_, f"w_school_{k}"] for s_, k in zip(c.student, c.school_slot)]
c["p"] = c.wh * c.ws; c["p"] = c.p / c.groupby("student").p.transform("sum")
tb = pd.DataFrame({"chosen school": [np.average(c.t, weights=c.p)], "all available zones": [np.average(alt.t, weights=alt.av > 0)]}, index=["mean car time (min)"]).round(1)
fig, ax = nt.panels(3, ncol=3, w=4.4, h=2.9)
ax[0].hist(c.t.clip(upper=90), bins=45, weights=c.p, color=nt.OBS); ax[0].set_xlabel("car time home - school (min)"); ax[0].set_ylabel("students")
b = (c.zone.map(dict(zip(ZS.zone, ZS.area_group.astype(str)))))
a = c.groupby(b).p.sum().reindex(nt.AREA_GROUPS).fillna(0); nt.bar(100 * a / a.sum(), "area group of the school zone", "share (%)", ax=ax[1], horizontal=True)
nt.bar(st.band.value_counts(), "enrolment band", "students", ax=ax[2])
plt.tight_layout(); plt.show()
print(f"possible home zones per student: {st.filter(like='w_home_').gt(0).sum(axis=1).value_counts().sort_index().to_dict()}; "
      f"possible school zones: {st.filter(like='w_school_').gt(0).sum(axis=1).value_counts().sort_index().to_dict()}")
tb

Car time from home to the chosen school (weighted over the possible home and school zones), area of the school zone and enrolment band.

In [ ]:
# car time to the chosen school zone against all available zones (chosen weighted by the location weights)
nt.chosen_vs_available([("Car time home - school (min)", np.repeat(c.t.values, np.maximum(1, (c.p * 20).round().astype(int).values)).clip(max=90),
                         alt.t[alt.av > 0].clip(upper=90).values)], "Chosen school zone against the available zones, estimation students")

How to read it: line = chosen school zones, grey = all available zones; the further apart the two, the more car time separates the choices.

In [ ]:
# Students (age, sex, income group, cars) and their school and home zones
fig, ax = nt.panels(5, ncol=5, w=3, h=2.7)
nt.hist(st.age, "age (years)", "students", bins=range(0, 60, 1), ax=ax[0])
nt.bar((100 * st.female.value_counts(normalize=True)).sort_index(), "female (1 yes)", "share (%)", ax=ax[1])
nt.bar((100 * st.income_group.value_counts(normalize=True)).sort_index(), "income group", "share (%)", ax=ax[2])
car = pd.Series(np.select([st.car0 == 1, st.carlow == 1], ["no car", "cars < drivers"], "enough cars")).value_counts()
nt.bar(100 * car / car.sum(), "cars", "share (%)", ax=ax[3], rot=20)
z = alt[alt.av > 0].drop_duplicates("zone").zone.nunique(); ax[4].axis("off"); ax[4].text(0, 0.5, f"{z:,} zones with enrolment\nof the segment's bands", fontsize=10)
plt.tight_layout(); plt.show()
SZ = pd.read_parquet(os.path.join(SH, "school_choice_work", "zone_size.parquet"))
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
nt.zone_map(c.groupby("zone").p.sum(), "students (school zone)", ax=ax[0], log=True)
hz_ = st.melt(id_vars="student", value_vars=[x for x in st.columns if x.startswith("home_zone_")], value_name="zone")
nt.zone_map(hz_[hz_.zone > 0].zone.value_counts(), "students (possible home zones)", ax=ax[1], log=True)
ax[0].set_title("schools", fontsize=10); ax[1].set_title("homes", fontsize=10)
plt.tight_layout(); plt.show()

Students (age, sex, income group, cars) and their school and home zones.

In [ ]:
# Bands and income groups of the two halves
sth = pd.read_parquet(os.path.join(os.path.dirname(M.DATA).replace(os.sep + "estimation", os.sep + "holdout"), os.path.basename(M.DATA).replace("_estimation", "_holdout") + "_students.parquet"))
pd.concat({"enrolment band": nt.half_comparison(st.band, sth.band, "x"), "income group": nt.half_comparison(st.income_group, sth.income_group, "x")})

Bands and income groups of the two halves.

In [ ]:
# Descriptive statistics
person = nt.explanatory(st[[c for c in st.columns if not c.startswith(("home_zone", "school_zone", "w_home", "w_school"))]])
alt_x = nt.explanatory(alt.drop(columns=["av"]))      # used in section 6

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: the chosen school zone (spread over the student's possible home and school zones by their weights) against all zones with enrolment of the student's band (each student counts once); the size term with the size weights at their estimates.

In [ ]:
# every explanatory variable of the specification: chosen against available school zones
import varplots as vp
VP, NST = vp.school_panels(M, st, alt, cand, vp.silo_labels(MODEL_FOLDER), dict(zip(rd.reference(MODEL_FOLDER)[0].parameter, rd.reference(MODEL_FOLDER)[0].coefficient)))
print(f"{len(VP)} variables, {NST:,} students")
vp.location_figures(VP, what="available", unit="students")

How to read it: dark grey line = the chosen alternatives, light bars = the available alternatives, as a share of each (%); dashed lines are the two means; 0/1 variables are shown as the share with value 1. A continuous variable that is zero at every alternative of a record cannot affect that record's choice, so such records are left out of its panel (e.g. an interaction with a person group; the share kept is in the panel title). The further apart the two, the more the variable separates the choices.

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The specification is the folder's script `model/estimate_school_location_high_school.py`, imported as it is. The start values are the article estimates.

In [ ]:
# Model specification
print(inspect.getsource(M.model))

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

Correlations of the variables that enter the utility, computed over the alternatives available to each decision maker
(alternative-specific variables) and over decision makers (person and household variables). |r| ≥ 0.7 and VIF ≥ 10 are
listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
Ca = nt.corr_heatmap(alt_x, "School-zone variables (sampled alternatives)")
display(nt.high_pairs(Ca)); display(nt.vif(alt_x))
Cp = nt.corr_heatmap(person, "Student variables"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the article coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the log-likelihood at the start values (the article estimates). Target: article -1217.0,
Biogeme -1216.9605 (N = 668, K = 6).

In [ ]:
# Log-likelihood at the article coefficients
if RUN_CHECK or RUN_ESTIMATION:
    t0 = time.time(); database, ll = M.model(); print(f"database built in {time.time()-t0:.0f} s")

if RUN_CHECK:
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(__import__("tempfile").mkdtemp())
    t0 = time.time()
    b = bio.BIOGEME(database, ll, parameters=p); b.model_name = NAME + "_check"
    ll0 = b.calculate_init_likelihood()
    os.chdir(here)
    print(f"N {database.num_rows():,}   LL at the article coefficients: {ll0:,.4f}   (target -1216.9605)   {time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The same call as `model/estimate_school_location_high_school.py`: Biogeme with its defaults (read from `output/biogeme.toml`), start values = article estimates.
Report written to `output/school_location_high_school.html`, `.yaml`, and `output/school_location_high_school_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); os.chdir(OUT)
    t0 = time.time()
    b = bio.BIOGEME(database, ll); b.model_name = NAME
    results = b.estimate()
    os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s"); print(results.short_summary())
    nt.estimates_table(results)[["parameter", "coefficient", "robust_se", "t", "p"]].rename(
        columns={"coefficient": "Value", "robust_se": "Robust std err.", "t": "Robust t-stat.", "p": "Robust p-value", "parameter": "Name"}
    ).to_csv(os.path.join(OUT, f"{NAME}_estimates.csv"), index=False)
else:
    ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
    y = [f for f in sorted(glob.glob(os.path.join(OUT, f"{NAME}*.yaml")), key=os.path.getmtime) if os.path.getmtime(f) > ref_time and "_check" not in f]
    results = EstimationResults.from_yaml_file(y[-1]) if y else None
    print(f"RUN_ESTIMATION = False: read {os.path.basename(y[-1])} (newer than model/reference_estimates.csv)" if y else "RUN_ESTIMATION = False and no current result in output/: sections 9-10 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term, from your estimates (from the article estimates if there are none). Income interactions are given at $25k, $50k and $100k; odds change = exp(coefficient) per unit.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
est_v = est_t if results is not None else rd.reference(MODEL_FOLDER)[0]
display(rd.readings(MODEL_FOLDER, est_v).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(fg.coef_rows(MODEL_FOLDER, est_v)); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "N": database.num_rows() if "database" in dir() else None,
                                                   "K": results.number_of_free_parameters})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    import tempfile
    est_data = M.DATA
    M.DATA = os.path.join(FOLDER, "input", "data", "holdout", 'school_hs_holdout')
    at = "your estimates" if results is not None else "the article coefficients"
    try:
        hdb, hll = M.model(start=results.get_beta_values() if results is not None else None)
    finally:
        M.DATA = est_data
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(tempfile.mkdtemp())
    b = bio.BIOGEME(hdb, hll, parameters=p); b.model_name = NAME + "_holdout"
    llh = b.calculate_init_likelihood(); os.chdir(here)
    print(f"hold-out: N {hdb.num_rows():,}   LL at {at} {llh:,.2f}   mean LL per observation {llh/hdb.num_rows():.4f}")
    if results is not None:
        print(f"estimation sample: mean LL per observation {results.final_loglikelihood/database.num_rows():.4f}")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out check by area

Observed and model shares of the hold-out records by area (model share = sum of the predicted probabilities). The map and its tables are in `output/holdout_map.png`, `output/holdout_puma.csv`, `output/holdout_county.csv`.

In [ ]:
# hold-out map (copied into output/) and its reading, numbers from output/holdout_*.csv
nt.show_png(os.path.join(OUT, "holdout_map.png"))
display(Markdown(rd.holdout_md(MODEL_FOLDER, OUT, os.path.join(FOLDER, "..", "00_shared_data", "zone_system", "zoneSystem.csv"))))

In [ ]:
# observed against model share by county (hold-out); the counties with the largest differences are labelled
HC = pd.read_csv(os.path.join(OUT, "holdout_county.csv")); HC = HC[HC.band == 'hs']; u = "Students"
nt.holdout_scatter(HC, names=rd.county_names(os.path.join(FOLDER, "..", "00_shared_data", "zone_system", "zoneSystem.csv")), unit=u, title="(a) Counties, hold-out")

How to read it: one point per county; on the line the share is predicted exactly, above the line the model is too high.

## Glossary

- **zone**: A traffic analysis zone of the model (1,588 zones).
- **size term**: mu x ln(size of the zone): with mu = 1 the probability is proportional to the zone's jobs or enrolment.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **BIC**: Bayesian information criterion: -2 log-likelihood + K ln N. Lower is better; it penalises extra parameters.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.